# 09쪽 · Chain: 요청까지만

## 준비
아래 셀은 파일 경로와 `.env` 로더를 준비합니다. 한 번 실행하고 넘어가세요.

In [ ]:
from pathlib import Path
import sys

# 노트북을 연 위치가 ipynb 폴더이면 한 단계 위로 이동합니다.
ROOT = Path.cwd()

if ROOT.name == "ipynb":
    ROOT = ROOT.parent

if not (ROOT / "lab" / "config.py").exists():
    raise RuntimeError("Langgraph_6_Week_Code 또는 ipynb 폴더에서 실행하세요.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lab.config import make_llm, load_settings

## 1. 준비

이 파일 안에서 필요한 도구를 직접 정의합니다.

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, MessagesState, START, END


def add_numbers(a: int, b: int) -> int:
    """두 정수 a와 b를 더합니다."""
    return a + b


def multiply_numbers(a: int, b: int) -> int:
    """두 정수 a와 b를 곱합니다."""
    return a * b

## 2. 모델에 도구 연결

키는 .env에서 읽습니다.

In [ ]:
llm = make_llm()

tools = [add_numbers, multiply_numbers]
tool_llm = llm.bind_tools(
    tools,
    parallel_tool_calls=False,
)

## 3. 모델을 호출하는 Node

현재 메시지를 보내고 새 응답을 반환합니다.

In [ ]:
def tool_calling_llm(state: MessagesState):
    response = tool_llm.invoke(state["messages"])

    return {"messages": [response]}

## 4. 그래프 연결

ToolNode는 없습니다.

In [ ]:
builder = StateGraph(MessagesState)

builder.add_node("tool_calling_llm", tool_calling_llm)

builder.add_edge(START, "tool_calling_llm")
builder.add_edge("tool_calling_llm", END)

graph = builder.compile()

## A. 인사 입력

계산 요청과 분리해서 실행합니다.

In [ ]:
messages = [
    HumanMessage(content="안녕하세요!"),
]

result = graph.invoke({"messages": messages})

for message in result["messages"]:
    message.pretty_print()

**결과 해설**

도구 호출 강제 설정이 없으므로 모델이 직접 답변할 수 있습니다.

## B. 덧셈 입력

같은 그래프에 별도의 새 입력을 보냅니다.

In [ ]:
messages = [
    HumanMessage(content="10과 20을 더해줘. 덧셈 도구를 사용해줘."),
]

result = graph.invoke({"messages": messages})

for message in result["messages"]:
    message.pretty_print()

**결과 해설**

도구를 요청해도 AIMessage의 요청에서 끝납니다. 실제 도구 실행은 없습니다.